In [1]:
import pandas as pd
import numpy as np

Load validated dataset

In [2]:
file_path = "../data/processed/life_expectancy_validated.csv"

df = pd.read_csv(file_path)

print("Dataset loaded!")
print("Shape:", df.shape)

Dataset loaded!
Shape: (17225, 51)


Check missing values

In [3]:
missing = df.isnull().sum()

missing = missing[missing > 0].sort_values(
    ascending=False
)

missing

Physicians                        11696
Tuberculosis_Incidence            11593
Clean_Cooking_Access              11561
Government_Health_Expenditure     11550
Health_Expenditure_Per_Capita     11536
Health_Expenditure_GDP            11533
OutOfPocket_Health_Expenditure    11533
Hospital_Beds                     11367
HIV_Prevalence                    11090
Basic_Sanitation_Access           10887
Basic_Water_Access                10852
HepB_Immunization                 10687
Energy_Use_Per_Capita             10537
Internet_Users                    10371
Tertiary_Enrollment                9883
Employment_Rate                    9278
Unemployment_Rate                  9278
Youth_Unemployment                 9278
Electricity_Access                 9128
Secondary_Enrollment               8927
PM25_Air_Pollution                 8827
Maternal_Mortality                 7826
Primary_Enrollment                 7696
Measles_Immunization               7224
DPT_Immunization                   7101


Remove rows where target is missing

In [4]:
df = df.dropna(
    subset=["Life_Expectancy"]
)

print("Shape after removing missing target:", df.shape)

Shape after removing missing target: (17126, 51)


Sort data

In [5]:
df = df.sort_values(
    ["Country", "Year"]
).reset_index(drop=True)

print("Data sorted successfully!")

Data sorted successfully!


Handle missing numeric values

In [6]:
numeric_columns = df.select_dtypes(
    include=np.number
).columns

df[numeric_columns] = (
    df.groupby("Country")[numeric_columns]
    .transform(lambda x: x.ffill().bfill())
)

Remaining missing values

In [7]:
for col in numeric_columns:
    df[col] = df[col].fillna(
        df[col].median()
    )

In [8]:
remaining_missing = df.isnull().sum()

remaining_missing = remaining_missing[
    remaining_missing > 0
]

remaining_missing

Country_Code    260
dtype: int64

Check outliers using IQR

In [9]:
numeric_columns = df.select_dtypes(include=np.number).columns

outlier_report = []

for col in numeric_columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = ((df[col] < lower) | (df[col] > upper)).sum()

    outlier_report.append({
        "Column": col,
        "Outlier_Count": outliers,
        "Outlier_Percentage": round(
            outliers / len(df) * 100, 2
        )
    })

outlier_report = pd.DataFrame(outlier_report)

outlier_report = outlier_report.sort_values(
    "Outlier_Count",
    ascending=False
)

outlier_report

,Column,Outlier_Count,Outlier_Percentage
10,Population,3249,18.97
24,HIV_Prevalence,3097,18.08
32,GDP,3037,17.73
38,Primary_Enrollment,2801,16.36
45,Internet_Users,2756,16.09
34,GNI_Per_Capita,2522,14.73
18,Health_Expenditure_Per_Capita,2470,14.42
30,GDP_Per_Capita,2449,14.30
31,GDP_Per_Capita_Constant,1982,11.57
5,Maternal_Mortality,1636,9.55
